In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

DATA_PATH = '/content/drive/MyDrive/GAMEEMO_processed/gameemo_preprocessed.pkl'
SAVE_DIR = '/content/drive/MyDrive/GAMEEMO_processed/results'
os.makedirs(SAVE_DIR, exist_ok=True)

with open(DATA_PATH, 'rb') as f:
    data = pickle.load(f)

X = data['X']
y = data['y']
subject_ids = data['subject_ids']
n_classes = len(data['label_names'])

trial_ids = np.array([f'{s}_{l}' for s, l in zip(subject_ids, y)])

print('X shape:', X.shape)
print('classes:', data['label_names'])

X shape: (8288, 512, 14)
classes: ['boring', 'calm', 'horror', 'funny']


In [ ]:
def get_subject_dependent_split(X, y, trial_ids, test_size=0.2, seed=42):
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_idx, test_idx = next(splitter.split(X, y, groups=trial_ids))
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


def get_cross_subject_split(X, y, subject_ids, test_size=0.2, seed=42):
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_idx, test_idx = next(splitter.split(X, y, groups=subject_ids))
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


X_train_sd, X_test_sd, y_train_sd, y_test_sd = get_subject_dependent_split(X, y, trial_ids)
X_train_cs, X_test_cs, y_train_cs, y_test_cs = get_cross_subject_split(X, y, subject_ids)

print('subject dependent train/test:', X_train_sd.shape, X_test_sd.shape)
print('cross subject train/test:', X_train_cs.shape, X_test_cs.shape)

subject dependent train/test: (6586, 512, 14) (1702, 512, 14)
cross subject train/test: (6512, 512, 14) (1776, 512, 14)


In [ ]:
# warmup schedule: ramp lr up for the first few epochs, then decay
# this is the standard fix for transformers that get stuck at epoch 1

class WarmupCosineSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak_lr, warmup_steps, total_steps):
        super().__init__()
        self.peak_lr = peak_lr
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup = self.peak_lr * step / tf.cast(self.warmup_steps, tf.float32)
        cosine = self.peak_lr * 0.5 * (
            1.0 + tf.cos(
                np.pi * (step - self.warmup_steps) /
                tf.cast(self.total_steps - self.warmup_steps, tf.float32)
            )
        )
        return tf.where(step < self.warmup_steps, warmup, cosine)

    def get_config(self):
        return {
            'peak_lr': self.peak_lr,
            'warmup_steps': self.warmup_steps,
            'total_steps': self.total_steps
        }

In [ ]:
def positional_encoding(length, depth):
    positions = np.arange(length)[:, np.newaxis]
    dims = np.arange(depth)[np.newaxis, :]
    angles = positions / np.power(10000, (2 * (dims // 2)) / np.float32(depth))
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[np.newaxis, :, :], dtype=tf.float32)


def transformer_encoder_block(x, num_heads, ff_dim, dropout_rate=0.3):
    key_dim = max(1, x.shape[-1] // num_heads)
    attn_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
    attn_output = layers.Dropout(dropout_rate)(attn_output)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attn_output)

    ff_output = layers.Dense(ff_dim, activation='gelu')(x)
    ff_output = layers.Dense(x.shape[-1])(ff_output)
    ff_output = layers.Dropout(dropout_rate)(ff_output)
    x = layers.LayerNormalization(epsilon=1e-6)(x + ff_output)
    return x


def build_eeg_transformer(n_timepoints, n_channels, n_classes,
                           d_model=32, num_heads=4, ff_dim=64,
                           num_transformer_blocks=2, dropout_rate=0.3):

    inputs = layers.Input(shape=(n_timepoints, n_channels))

    # project channels up to d_model and downsample time
    x = layers.Conv1D(d_model, kernel_size=7, strides=4, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv1D(d_model, kernel_size=5, strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    seq_len = x.shape[1]
    x = x + positional_encoding(seq_len, d_model)

    for _ in range(num_transformer_blocks):
        x = transformer_encoder_block(x, num_heads=num_heads, ff_dim=ff_dim, dropout_rate=dropout_rate)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(n_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    return model

In [ ]:
def train_and_evaluate(model, X_train, y_train, X_test, y_test,
                       model_name, protocol_name, epochs=60, batch_size=32):

    steps_per_epoch = len(X_train) // batch_size
    total_steps = epochs * steps_per_epoch
    warmup_steps = 5 * steps_per_epoch  # warm up for 5 epochs

    schedule = WarmupCosineSchedule(
        peak_lr=3e-4,
        warmup_steps=warmup_steps,
        total_steps=total_steps
    )
    optimizer = tf.keras.optimizers.Adam(learning_rate=schedule)

    model.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    checkpoint_path = os.path.join(SAVE_DIR, f'{model_name}_{protocol_name}_best.keras')

    callback_list = [
        callbacks.EarlyStopping(patience=15, restore_best_weights=True, monitor='val_accuracy'),
        callbacks.ModelCheckpoint(checkpoint_path, save_best_only=True, monitor='val_accuracy'),
    ]

    history = model.fit(
        X_train, y_train,
        validation_split=0.15,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callback_list,
        verbose=1
    )

    y_pred = np.argmax(model.predict(X_test), axis=1)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')

    print(f'{model_name} ({protocol_name}) accuracy: {acc:.4f}, f1: {f1:.4f}')
    print(classification_report(y_test, y_pred, target_names=data['label_names'],
                                zero_division=0))

    result = {
        'model': model_name,
        'protocol': protocol_name,
        'accuracy': acc,
        'f1_score': f1,
        'confusion_matrix': confusion_matrix(y_test, y_pred).tolist(),
        'history': history.history
    }

    return result, model

# Transformer Subject Dependent

In [ ]:
transformer_model = build_eeg_transformer(n_timepoints=512, n_channels=14, n_classes=n_classes)
transformer_model.summary()

result_transformer_sd, transformer_sd = train_and_evaluate(
    transformer_model, X_train_sd, y_train_sd, X_test_sd, y_test_sd,
    model_name='eeg_transformer', protocol_name='subject_dependent'
)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 512, 14)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 128, 32)   │      3,168 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128, 32)   │        128 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 128, 32)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 64, 32)    │      5,152 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 32)    │        128 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 64, 32)    │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 64, 32)    │          0 │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 64, 32)    │      4,224 │ add[0][0],        │
│ (MultiHeadAttentio… │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64, 32)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 64, 32)    │          0 │ add[0][0],        │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 64, 32)    │         64 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64, 64)    │      2,112 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64, 32)    │      2,080 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64, 32)    │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 64, 32)    │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 64, 32)    │         64 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 64, 32)    │      4,224 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 64, 32)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 28,036 (109.52 KB)

 Trainable params: 27,908 (109.02 KB)

 Non-trainable params: 128 (512.00 B)

Epoch 1/60
175/175 ━━━━━━━━━━━━━━━━━━━━ 33s 82ms/step - accuracy: 0.2640 - loss: 1.3991 - val_accuracy: 0.2480 - val_loss: 1.3914
Epoch 2/60
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.2865 - loss: 1.3817 - val_accuracy: 0.2368 - val_loss: 1.3971
Epoch 3/60
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3335 - loss: 1.3501 - val_accuracy: 0.2247 - val_loss: 1.4040
Epoch 4/60
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.4039 - loss: 1.2691 - val_accuracy: 0.3289 - val_loss: 1.3589
Epoch 5/60
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.4757 - loss: 1.1422 - val_accuracy: 0.3968 - val_loss: 1.3083
Epoch 6/60
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5522 - loss: 1.0167 - val_accuracy: 0.4443 - val_loss: 1.2757
Epoch 7/60
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.6245 - loss: 0.8986 - val_accuracy: 0.4049 - val_loss: 1.6156
Epoch 8/60
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.6867 - loss: 0.7889 - val_accuracy

# Transformer Cross Subject

In [ ]:
transformer_model_2 = build_eeg_transformer(n_timepoints=512, n_channels=14, n_classes=n_classes)

result_transformer_cs, transformer_cs = train_and_evaluate(
    transformer_model_2, X_train_cs, y_train_cs, X_test_cs, y_test_cs,
    model_name='eeg_transformer', protocol_name='cross_subject'
)

Epoch 1/60
173/173 ━━━━━━━━━━━━━━━━━━━━ 30s 79ms/step - accuracy: 0.2524 - loss: 1.4431 - val_accuracy: 0.2088 - val_loss: 1.4011
Epoch 2/60
173/173 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.2683 - loss: 1.4018 - val_accuracy: 0.2293 - val_loss: 1.3854
Epoch 3/60
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3281 - loss: 1.3573 - val_accuracy: 0.2897 - val_loss: 1.3739
Epoch 4/60
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.4042 - loss: 1.2684 - val_accuracy: 0.3838 - val_loss: 1.2733
Epoch 5/60
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.4918 - loss: 1.1200 - val_accuracy: 0.4319 - val_loss: 1.1350
Epoch 6/60
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5713 - loss: 0.9878 - val_accuracy: 0.4841 - val_loss: 1.1489
Epoch 7/60
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6179 - loss: 0.8913 - val_accuracy: 0.5599 - val_loss: 0.9743
Epoch 8/60
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6625 - loss: 0.8096 - val_accuracy:

# Save Result

In [ ]:
day4_results = {
    'transformer_subject_dependent': result_transformer_sd,
    'transformer_cross_subject': result_transformer_cs,
}

results_path = os.path.join(SAVE_DIR, 'day4_results.pkl')
with open(results_path, 'wb') as f:
    pickle.dump(day4_results, f)

print('saved day 4 results to', results_path)

for key, res in day4_results.items():
    print(key, '- accuracy:', round(res['accuracy'], 4), 'f1:', round(res['f1_score'], 4))

saved day 4 results to /content/drive/MyDrive/GAMEEMO_processed/results/day4_results.pkl
transformer_subject_dependent - accuracy: 0.7832 f1: 0.7887
transformer_cross_subject - accuracy: 0.8902 f1: 0.8898
